# Neural Network Models for Time Series Classification

This notebook demonstrates the process of loading, preprocessing, and evaluating different neural network architectures for time series classification. Specifically, it focuses on Convolutional Neural Networks (CNN), Recurrent Neural Networks (RNN) (using LSTM), and an Autoencoder-based classifier.

The workflow includes:
1.  **Data Loading**: Loading pre-split training and testing data from a `.npz` file.
2.  **Dataset and DataLoader Preparation**: Creating custom PyTorch `Dataset` and `DataLoader` instances to efficiently manage and batch the time series data, including a validation split.
3.  **Helper Functions**: Defining utility functions, such as `compute_accuracy`, for model evaluation.
4.  **Model Evaluation**: Loading pre-trained CNN, RNN (LSTM), and Autoencoder models and evaluating their performance on the test dataset.
5.  **Conclusion**: Summarizing the performance of the evaluated models.

The goal is to compare the effectiveness of these different approaches for time series classification on a given dataset.

# **Loading the Data**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### **Importing Necessary Libraries**
This section imports the core libraries required for numerical operations (`numpy`), data manipulation (`pandas`), plotting (`matplotlib`), and building/training neural networks (`torch`).

In [ ]:
import torch

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset

In [ ]:
project_data = np.load("/content/project_data.npz", allow_pickle=True)

X_train_np = project_data['X_train']
y_train_np = project_data['y_train']
X_test_np = project_data['X_test']
y_test_np = project_data['y_test']

### **Loading Raw Data**
Here, the time series data is loaded from a `.npz` file. This file is assumed to contain pre-split `X_train`, `y_train`, `X_test`, and `y_test` arrays, representing the features and labels for training and testing, respectively.

In [ ]:
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(DEVICE)

cpu


### **Device Configuration**
This cell determines whether a GPU (CUDA) is available for computations. If a GPU is present, PyTorch will utilize it; otherwise, operations will default to the CPU. This ensures efficient computation based on available hardware.

In [ ]:
class My_Dataset(Dataset):
  def __init__(self, X, y):
    self.X = torch.tensor(X, dtype=torch.float32).to(DEVICE)
    self.y = torch.tensor(y, dtype=torch.long).to(DEVICE)

  def __len__(self):
    return len(self.X)

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

### **Custom PyTorch Dataset**
This `My_Dataset` class inherits from `torch.utils.data.Dataset` and is designed to handle the time series data. It converts numpy arrays to PyTorch tensors and moves them to the appropriate device (CPU or GPU), making them ready for use with PyTorch's `DataLoader`.

In [ ]:
train_dataset = My_Dataset(X_train_np, y_train_np)
test_dataset = My_Dataset(X_test_np, y_test_np)

### **Instantiating Datasets**
Here, instances of `My_Dataset` are created for both the training and testing data, wrapping the loaded numpy arrays into a PyTorch-compatible format.

In [ ]:
print(len(train_dataset))
print(len(test_dataset))

40500
4500


### **DataLoaders Configuration**
This section prepares the `DataLoader` instances. It first splits the training dataset into training and validation sets (90/10 split). Then, `DataLoader` objects are created for the training, validation, and test sets. These data loaders manage batching, shuffling (for training), and loading data efficiently during model training and evaluation.

# **DataLoaders**

In [ ]:
from torch.utils.data import DataLoader

validation = int(0.1 * len(train_dataset))
train_dataset, validation_dataset = torch.utils.data.random_split(train_dataset, [len(train_dataset) - validation, validation])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
valid_loader = DataLoader(validation_dataset, batch_size=32, shuffle=False)

In [ ]:
print(len(train_loader))
print(len(valid_loader))
print(len(test_loader))

1140
127
141


### **Verifying DataLoader Lengths**
These print statements confirm the number of batches in each DataLoader, providing an overview of the data distribution after batching.

In [ ]:
len(train_loader.dataset)

36450

### **Global Parameters**
Defines `NUM_EPOCHS`, a global constant for the number of training epochs, which will be used during model training.

# **Helper Functions**

In [ ]:
NUM_EPOCHS = 25

In [ ]:
def compute_accuracy(model, data_loader, device):

    with torch.no_grad():

        correct_pred, num_examples = 0, 0

        for i, (features, targets) in enumerate(data_loader):

            features = features.to(device)
            targets = targets.float().to(device)

            logits = model(features)
            _, predicted_labels = torch.max(logits, 1)

            num_examples += targets.size(0)
            correct_pred += (predicted_labels == targets).sum()
    return correct_pred.float()/num_examples * 100

### **Accuracy Computation Function**
This helper function, `compute_accuracy`, calculates the classification accuracy of a given model on a specified dataset (via `DataLoader`). It operates in `no_grad()` mode to prevent unnecessary gradient calculations during evaluation.

# **Models with Test Accuracies**

# **CNN**

In [ ]:
class CNN(torch.nn.Module):
  def  init (self, num_classes):
    super(CNN, self). init ()
    self.conv = torch.nn.Sequential(
      torch.nn.Conv1d(8, 32, kernel_size=3, stride=1, padding=1),
      torch.nn.BatchNorm1d(32),
      torch.nn.ReLU(),
      torch.nn.MaxPool1d(kernel_size=2, stride=2),
      torch.nn.Conv1d(32, 64, kernel_size=3, stride=1, padding=1),
      torch.nn.BatchNorm1d(64),
      torch.nn.ReLU(),
      torch.nn.MaxPool1d(kernel_size=2, stride=2),
      torch.nn.AdaptiveAvgPool1d(1),
      )
    self.classifier = torch.nn.Sequential(
      torch.nn.Flatten(),
      torch.nn.Linear(64, 32),
      torch.nn.ReLU(),
      torch.nn.Linear(32, num_classes)
      )

  def forward(self, x):
    x = x.permute(0, 2, 1) #to convert(batch, seq_len, features) -> (batch, features, seq_len)
    x = self.conv(x)
    x = self.classifier(x)
    return x

### **Convolutional Neural Network (CNN) Model**
This section defines the architecture for the CNN model. It typically consists of convolutional layers for feature extraction, followed by pooling layers for dimensionality reduction, and finally fully connected layers for classification. The `forward` method describes how data flows through the network.

In [ ]:
cnn = torch.load('cnn_best_model.pt',weights_only=False)

cnn.eval()

CNN(
  (conv): Sequential(
    (0): Conv1d(8, 32, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): AdaptiveAvgPool1d(output_size=1)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=64, out_features=32, bias=True)
    (2): ReLU()
    (3): Linear(in_features=32, out_features=11, bias=True)
  )
)

### **Recurrent Neural Network (RNN) Model (LSTM)**
This section defines the architecture for the RNN model, specifically using a Long Short-Term Memory (LSTM) layer. LSTMs are well-suited for sequence data like time series due to their ability to capture long-term dependencies. The model processes the input sequence and uses a final fully connected layer for classification.

In [ ]:
test_acc = compute_accuracy(cnn, test_loader, device=DEVICE)
print(f'Test accuracy {test_acc :.2f}%')

Test accuracy 94.16%


# **RNN**

In [ ]:
class RNN(torch.nn.Module):
  def   init  (self, input_size, hidden_size, output_size):
    super(RNN, self).  init  ()
    self.hidden_size = hidden_size
    self.lstm = torch.nn.LSTM(input_size, hidden_size, batch_first=True)
    self.fc = torch.nn.Linear(hidden_size, output_size)

  def forward(self, x):
    x, h = self.lstm(x)
    x = self.fc(x[:, -1, :])
    return x

### **Autoencoder Model**
This section is reserved for the Autoencoder model definition. An Autoencoder is an unsupervised learning model that learns an efficient encoding of the input data. For classification, its encoded representation is often passed to a classifier (which might be part of the loaded model). While the class definition is not explicitly provided here, the subsequent cells load and evaluate a pre-trained Autoencoder-based classifier.

In [ ]:
rnn = torch.load('rnn_best_model.pt',weights_only=False)

rnn.eval()

RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.

In [ ]:
test_acc = compute_accuracy(rnn, test_loader, device=DEVICE)
print(f'Test accuracy {test_acc :.2f}%')

# **Autoencoder**

In [ ]:
ae = torch.load('autoencoder_best_model.pt',weights_only=False)

ae.eval()

In [ ]:
test_acc = compute_accuracy(ae, test_loader, device=DEVICE)
print(f'Test accuracy {test_acc :.2f}%')

Out of the 3 models the best accuracy was acheived by the Autoencoder-Classifier Model.

### **Model Performance Summary**
This concluding section summarizes the test accuracies achieved by the CNN, RNN, and Autoencoder models, highlighting which model performed best on the given time series classification task.